# Stage 3: Real-Time Streaming Pipeline

Project Topic: SupplyChain Manufacturing

This notebook implements the real-time streaming component of the PrecisionParts analytics platform. While Stage 2 processed historical data in batch, Stage 3 handles live equipment sensor telemetry.

The pipeline:
1. Reads raw sensor data to compute per-machine baselines (mean, std, 95th percentile)
2. Connects to the `machine-telemetry` Kafka topic
3. Parses and validates incoming sensor events
4. Computes 1-minute rolling averages per machine
5. Triggers alerts when readings deviate from data-driven baselines

## 1. SparkSession Initialization

In [12]:
import subprocess
subprocess.run(['pip', 'install', 'kafka-python', 'pyspark'], check=True)
print('Dependencies installed.')

Dependencies installed.


In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder \
    .appName('SupplyChain-Stage3-Streaming') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '1g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('SparkSession initialized successfully.')

Spark version: 3.5.0
SparkSession initialized successfully.


## 2. Path Configuration

In [14]:
RAW_SENSORS   = '../data/raw/equipment-sensors.csv.gz'
KAFKA_BROKER  = 'localhost:9092'
TOPIC         = 'machine-telemetry'
ALERTS_OUTPUT = '../data/analytics/streaming_alerts'

print('Configuration:')
print(f'  Raw sensors:    {RAW_SENSORS}')
print(f'  Kafka broker:   {KAFKA_BROKER}')
print(f'  Kafka topic:    {TOPIC}')
print(f'  Alerts output:  {ALERTS_OUTPUT}')

Configuration:
  Raw sensors:    ../data/raw/equipment-sensors.csv.gz
  Kafka broker:   localhost:9092
  Kafka topic:    machine-telemetry
  Alerts output:  ../data/analytics/streaming_alerts


## 3. Compute Machine Baselines

Stage 2 used fixed anomaly thresholds (`TEMP_THRESHOLD = 200.0`, `VIBRATION_THRESHOLD = 15.0`). Stage 3 improves on this by computing **per-machine, data-driven thresholds** from the historical sensor data. Each machine gets its own alert threshold based on its actual operating profile rather than a single global cutoff.

Alert conditions (from project spec):
- **Temperature alert**: rolling avg > `temp_mean + 2 x temp_std`
- **Vibration alert**: rolling avg > `vib_p95` (historical 95th percentile)

In [15]:
raw_sensors = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(RAW_SENSORS)
)

baselines_df = (
    raw_sensors
    .filter(F.col('temperature_c').isNotNull() & F.col('vibration_mm_s').isNotNull())
    .filter(F.col('temperature_c').between(-20, 500))
    .filter(F.col('vibration_mm_s') >= 0)
    .groupBy('machine_id')
    .agg(
        F.mean('temperature_c').alias('temp_mean'),
        F.stddev('temperature_c').alias('temp_std'),
        F.mean('vibration_mm_s').alias('vib_mean'),
        F.stddev('vibration_mm_s').alias('vib_std'),
        F.percentile_approx('vibration_mm_s', 0.95).alias('vib_p95'),
        F.count('machine_id').alias('reading_count')
    )
)

baselines_df.cache()
print(f'Baselines computed for {baselines_df.count()} machines.')
baselines_df.show(10, truncate=False)

Baselines computed for 800 machines.
+----------+------------------+------------------+------------------+------------------+-------+-------------+
|machine_id|temp_mean         |temp_std          |vib_mean          |vib_std           |vib_p95|reading_count|
+----------+------------------+------------------+------------------+------------------+-------+-------------+
|MACH-0001 |58.025487012986964|20.08912053799456 |2.141396103896106 |1.344565211769307 |4.01   |616          |
|MACH-0002 |58.27350000000001 |19.14799965796696 |2.155266666666666 |1.3103938908349093|3.89   |600          |
|MACH-0003 |58.97711598746088 |19.667709987672936|2.1663479623824453|1.3407761469463626|4.11   |638          |
|MACH-0004 |57.45260586319216 |20.16088353336813 |2.0987133550488606|1.349469646020605 |4.04   |614          |
|MACH-0005 |57.58805970149258 |19.18447555598445 |2.1100447761194023|1.303081032539535 |3.94   |670          |
|MACH-0006 |58.13048576214407 |19.661488656076767|2.136750418760467 |1.3153

26/05/02 21:14:35 WARN CacheManager: Asked to cache already cached data.        


## 4. Define Kafka Message Schema

The Kafka producer (`producers/event_producer.py`) sends one JSON message per sensor reading. The schema below matches the fields the producer emits.

In [16]:
telemetry_schema = StructType([
    StructField('machine_id',           StringType(), True),
    StructField('factory_id',           StringType(), True),
    StructField('timestamp',            StringType(), True),
    StructField('temperature_c',        DoubleType(), True),
    StructField('vibration_mm_s',       DoubleType(), True),
    StructField('power_consumption_kw', DoubleType(), True),
    StructField('oil_pressure_psi',     DoubleType(), True),
    StructField('status',               StringType(), True),
])

print('Telemetry schema defined.')
print('Fields:', [f.name for f in telemetry_schema.fields])

Telemetry schema defined.
Fields: ['machine_id', 'factory_id', 'timestamp', 'temperature_c', 'vibration_mm_s', 'power_consumption_kw', 'oil_pressure_psi', 'status']


## 5. Connect to Kafka and Parse Messages

In [17]:
raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BROKER)
    .option('subscribe', TOPIC)
    .option('startingOffsets', 'latest')
    .option('failOnDataLoss', 'false')
    .load()
)

parsed_stream = (
    raw_stream
    .selectExpr('CAST(value AS STRING) as json_str')
    .select(F.from_json(F.col('json_str'), telemetry_schema).alias('data'))
    .select('data.*')
    .withColumn('event_time', F.to_timestamp(F.col('timestamp')))
    .filter(
        F.col('machine_id').isNotNull() &
        F.col('temperature_c').isNotNull() &
        F.col('vibration_mm_s').isNotNull() &
        (F.col('temperature_c') > 0) &
        (F.col('vibration_mm_s') >= 0)
    )
)

print('Kafka stream connected and schema applied.')
parsed_stream.printSchema()

Kafka stream connected and schema applied.
root
 |-- machine_id: string (nullable = true)
 |-- factory_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- power_consumption_kw: double (nullable = true)
 |-- oil_pressure_psi: double (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## 6. Compute 1-Minute Rolling Averages

For each machine, we compute the average temperature and vibration over a 1-minute sliding window updated every 30 seconds. The 2-minute watermark allows Spark to handle slightly late-arriving sensor events without holding state indefinitely.

In [18]:
rolling_averages = (
    parsed_stream
    .withWatermark('event_time', '2 minutes')
    .groupBy(
        F.window(F.col('event_time'), '1 minute', '30 seconds'),
        F.col('machine_id'),
        F.col('factory_id')
    )
    .agg(
        F.avg('temperature_c').alias('avg_temp'),
        F.avg('vibration_mm_s').alias('avg_vibration'),
        F.avg('power_consumption_kw').alias('avg_power'),
        F.avg('oil_pressure_psi').alias('avg_oil_pressure'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'machine_id', 'factory_id',
        'avg_temp', 'avg_vibration', 'avg_power', 'avg_oil_pressure'
    )
)

print('Rolling averages defined (1-min window, 30-sec slide).')

Rolling averages defined (1-min window, 30-sec slide).


## 7. Anomaly Detection

Each rolling window is joined against the per-machine baselines from Section 3. Two alert types are flagged:

- **TEMP alert**: `avg_temp > temp_mean + 2 x temp_std`
- **VIB alert**: `avg_vibration > vib_p95`

Rows that exceed both thresholds are labeled **CRITICAL**. Rows that exceed one are labeled **WARNING**.

In [19]:
alerts_stream = (
    rolling_averages
    .join(baselines_df, on='machine_id', how='left')
    .withColumn('temp_threshold', F.round(F.col('temp_mean') + 2 * F.col('temp_std'), 2))
    .withColumn('vib_threshold',  F.round(F.col('vib_p95'), 2))
    .withColumn('temp_alert', F.when(F.col('avg_temp')      > F.col('temp_threshold'), True).otherwise(False))
    .withColumn('vib_alert',  F.when(F.col('avg_vibration') > F.col('vib_threshold'),  True).otherwise(False))
    .withColumn('alert_level',
        F.when(F.col('temp_alert') & F.col('vib_alert'), 'CRITICAL')
        .when(F.col('temp_alert') | F.col('vib_alert'), 'WARNING')
        .otherwise('NORMAL')
    )
    .filter(F.col('alert_level') != 'NORMAL')
    .select(
        'window_start', 'window_end', 'machine_id', 'factory_id',
        'avg_temp', 'temp_threshold', 'temp_alert',
        'avg_vibration', 'vib_threshold', 'vib_alert',
        'alert_level'
    )
)

print('Anomaly detection defined.')
print('Alert levels: WARNING (one threshold exceeded), CRITICAL (both exceeded)')

Anomaly detection defined.
Alert levels: WARNING (one threshold exceeded), CRITICAL (both exceeded)


## 8. Start Streaming Query

This cell starts the streaming query and prints alerts to the console as they are detected.

Before running this cell, start the Kafka producer in a **separate terminal**:
```
cd ~/Desktop/The_Leftside_Undergrads_Final/producers
python event_producer.py
```

In [ ]:
query = (
    alerts_stream
    .writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', 'false')
    .option('numRows', 20)
    .trigger(processingTime='30 seconds')
    .start()
)

print('Streaming query started. Waiting for alerts...')
print('Run the producer in a separate terminal to start sending data.')
query.awaitTermination(timeout=300)
print('Streaming query stopped.')

26/05/02 21:16:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/5j/llgjjm_x0mj5jrzhy2dxnjwm0000gn/T/temporary-622ead56-7303-4311-aa0b-70db728ee8bc. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/02 21:16:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/02 21:16:18 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Streaming query started. Waiting for alerts...
Run the producer in a separate terminal to start sending data.
-------------------------------------------
Batch: 0
-------------------------------------------
+------------+----------+----------+----------+--------+--------------+----------+-------------+-------------+---------+-----------+
|window_start|window_end|machine_id|factory_id|avg_temp|temp_threshold|temp_alert|avg_vibration|vib_threshold|vib_alert|alert_level|
+------------+----------+----------+----------+--------+--------------+----------+-------------+-------------+---------+-----------+
+------------+----------+----------+----------+--------+--------------+----------+-------------+-------------+---------+-----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------+----------+--------+--------------+----------+-------------+-------------+---------+-----------+
|window_start 

## 9. Write Alerts to Parquet (for Stage 4 Airflow DAG)

In [ ]:
import os
os.makedirs(ALERTS_OUTPUT, exist_ok=True)

alerts_to_disk = (
    alerts_stream
    .writeStream
    .outputMode('append')
    .format('parquet')
    .option('path', ALERTS_OUTPUT)
    .option('checkpointLocation', '/tmp/spark-checkpoints/alerts')
    .trigger(processingTime='30 seconds')
    .start()
)

print(f'Alerts being written to {ALERTS_OUTPUT}')

## Stage 3 Completion Checklist

- Installed Kafka and Spark Structured Streaming dependencies
- Computed per-machine baselines (mean, std, 95th percentile) from raw sensor data
- Defined Kafka topic schema matching the producer output
- Connected Spark Structured Streaming to the `machine-telemetry` Kafka topic
- Validated incoming sensor events (null checks, range filters)
- Computed 1-minute rolling averages per machine with 30-second slide
- Joined live stream with historical baselines
- Flagged WARNING alerts (one threshold exceeded) and CRITICAL alerts (both exceeded)
- Streamed alerts to console for real-time monitoring
- Wrote alerts to Parquet for downstream Stage 4 Airflow pipeline